In [ ]:
!pip -q install crewai duckduckgo-search ollama
!pip -q install 'crewai[tools]' decouple langchain-exa exa_py==1.0.7

import os
os.environ["OLLAMA_HOST"] = "http://localhost:11434"
os.environ["OLLAMA_API_KEY"] = "ollama"
LLM_MODEL = "ollama/mistral"
import urllib.request
import urllib.error
import json

def check_ollama_server():
    host = os.environ["OLLAMA_HOST"].rstrip("/")
    url = f"{host}/v1/models"
    try:
        with urllib.request.urlopen(url, timeout=5) as response:
            data = json.load(response)
            print("Ollama server erreichbar:", url)
            models = data.get("models", []) if isinstance(data, dict) else data
            print("Verfügbare Modelle:", [item.get("name") for item in models])
    except urllib.error.URLError as exc:
        raise RuntimeError(f"Konnte Ollama-Server nicht erreichen ({url}): {exc}")

check_ollama_server()



os.environ["EXA_API_KEY"] = 'YOUR_EXA_API_KEY'  # Replace with your actual Exa API key


from exa_py import Exa
from crewai.tools import tool

class ExaSearchTool:
	@tool
	def search(query: str):
		"""Search for a webpage based on the query."""
		return ExaSearchTool._exa().search(f"{query}", use_autoprompt=True, num_results=10)

	@tool
	def find_similar(url: str):
		"""Search for webpages similar to a given URL.
		The url passed in should be a URL returned from `search`.
		"""
		return ExaSearchTool._exa().find_similar(url,
		                                        	num_results=10,
		                                       	 # category="news"  # Specify the category you need from these - company, research paper, news, github, tweet, movie, song, personal site, and pdf
																						)

	@tool
	def get_contents(ids: str):
		"""Get the contents of a webpage.
		The ids must be passed in as a list, a list of ids returned from `search`.
		"""
		ids = eval(ids)
		contents = str(ExaSearchTool._exa().get_contents(ids))
		print(contents)
		contents = contents.split("URL:")
		contents = [content[:1000] for content in contents]
		return "\n\n".join(contents)

	def tools():
		return [ExaSearchTool.search, ExaSearchTool.find_similar, ExaSearchTool.get_contents]

	def _exa():
		return Exa(api_key=os.environ["EXA_API_KEY"])

from textwrap import dedent
from crewai import Agent

class OsintAgents():
	def CompanyInfo_agent(self):
		return Agent(
			role='Company Information Specialist',
			goal='Conduct thorough research on the company,its website, founded date, founders, Headquarter location, industry and Subsidiaries.',
			tools=ExaSearchTool.tools(),
			llm=LLM_MODEL,
			backstory=dedent("""\
					 As a Company Research Specialist, your mission is to uncover detailed information
           about the company, what is the company is all about, its official website and url of website, founded date of the company, founders names, location of the headquarter, industry of the company and its subsidiaries.
           Your insights will provide a comprehensive overview of the company's background."""),
			verbose=True
		)

	def WebsiteAnalysis_agent(self):
		return Agent(
			role='Website Analyst',
			goal='Analyze the official Website of the company its key information',
			tools=ExaSearchTool.tools(),
			llm=LLM_MODEL,
			backstory=dedent("""\
					As a Website Analyst, your analysis will focus on the company's website Structure like its sections and pages information, its proper metadata analysis, use tools like BuiltWith to identify Technology stack used in the company's website. Also find out the SSL/TLS configuration.
          Your will provide the detailed analysis of the companys website."""),
			verbose=True
		)

	def NetworkAnalysis_agent(self):
		return Agent(
			role='Domain and Network Analyst',
			goal='Analyze the Domain Registration Details, DNS Records, Subdomains, IP Address official Website and Network Services.',
			tools=ExaSearchTool.tools(),
			llm=LLM_MODEL,
			backstory=dedent("""\
          As Domain and Network Analyst, your analyis will focus on Domain Registration Details of the website use tools like whois, you will provide the DNS records using DNSdumster, you will provide the subdomains by using google dorking techniques,
          you will also provide the IP Address associated with the domain and provide the Network services by using the tool shodan."""),
			verbose=True
		)

	def SocialMediaAndContact_agent(self):
		return Agent(
			role='Social Media and Contact Information Specialist',
			goal='Gather detailed information about the company\'s social media presence and contact details, including phone numbers, email addresses, and key personnel contact information.',
			tools=ExaSearchTool.tools(),
			llm=LLM_MODEL,
			backstory=dedent("""\
          As a Social Media and Contact Information Specialist, your mission is to uncover the company\'s presence on social media platforms like LinkedIn, Facebook, Twitter, GitHub, Instagram, and others.
                Additionally, gather comprehensive contact information including phone numbers, email addresses, and any available contact details of key personnel working in the organization."""),
			verbose=True
		)

	def SearchEngineIntelligence_agent(self):
		return Agent(
			role='Search Engine Intelligence Specialist',
			goal='Use Google Dorking techniques to uncover hidden information like PDFs and confidential files, and find recent news articles about the company.',
			tools=ExaSearchTool.tools(),
			llm=LLM_MODEL,
			backstory=dedent("""\
          As a Search Engine Intelligence Specialist, your mission is to uncover hidden information about the company using Google Dorking techniques.
            You will also gather and analyze recent news articles about the company, deriving conclusions from the findings."""),
			verbose=True
		)

	def BusinessInformation_agent(self):
		return Agent(
			role='Business Information Specialist',
			goal='Gather comprehensive business information about the company including company overview, financial information, key personnel, and partnerships',
			tools=ExaSearchTool.tools(),
			llm=LLM_MODEL,
			backstory=dedent("""\
          As a Business Information Specialist, your mission is to gather detailed business information from various sources.
            You will provide an overview of the company, financial data, information about key personnel, and details of company partnerships."""),
			verbose=True
		)

	def RegulatoryLegalTechnicalFootprint_agent(self):
		return Agent(
			role='Regulatory, Legal, and Technical Footprint Specialist',
			goal='Gather information about the company’s regulatory filings, legal issues, security posture, and email patterns.',
			tools=ExaSearchTool.tools(),
			llm=LLM_MODEL,
			backstory=dedent("""\
          As a Regulatory, Legal, and Technical Footprint Specialist, your mission is to gather information on the company’s regulatory filings and legal issues, and assess its technical footprint.
            This includes identifying vulnerabilities and email patterns using various tools."""),
			verbose=True
		)

	def IntellectualProperty_agent(self):
		return Agent(
			role='Intellectual Property Specialist',
			goal='Gather information about the company’s patents, trademarks, and copyrights.',
			tools=ExaSearchTool.tools(),
			llm=LLM_MODEL,
			backstory=dedent("""\
          As an Intellectual Property Specialist, your mission is to uncover and document the company’s intellectual property assets.
            This includes registered patents, trademarks, and significant copyrights."""),
			verbose=True
		)

	def EmployeeHiringInformation_agent(self):
		return Agent(
			role='Employee and Hiring Information Specialist',
			goal='Gather information about current job listings and employee reviews of the company.',
			tools=ExaSearchTool.tools(),
			llm=LLM_MODEL,
			backstory=dedent("""\
          As an Employee and Hiring Information Specialist, your mission is to gather details about the company’s hiring practices and employee experiences.
            This includes current job openings and reviews from sites like Glassdoor."""),
			verbose=True
		)

	def CommunityPublicPerception_agent(self):
		return Agent(
			role='Community and Public Perception Specialist',
			goal='Gather customer reviews and forum discussions related to the company.',
			tools=ExaSearchTool.tools(),
			llm=LLM_MODEL,
			backstory=dedent("""\
            As a Community and Public Perception Specialist, your mission is to gather and analyze public opinions about the company.
            This includes customer reviews from various platforms and forum discussions."""),
			verbose=True
		)

	def OSINTReportGenerator_agent(self):
		return Agent(
      role='OSINT Report Generator',
      goal='Compile all gathered information into a detailed and comprehensive OSINT report.',
      tools=ExaSearchTool.tools(),
			llm=LLM_MODEL,
      backstory=dedent("""\
            As the OSINT Report Generator, your role is to consolidate the information from various agents, including Company Information, Website Analysis, Domain and Network Analysis, Social Media and Contact Information, Search Engine Intelligence, Business Information, Regulatory and Legal Information, Technical Footprint, Intellectual Property, Employee and Hiring Information, Community and Public Perception, and Dark Web Mentions, into a detailed and comprehensive OSINT report.
            This report will provide a thorough overview and analysis of the company."""),
      verbose=True
      )



from crewai import Task

class OsintAnalysisTask():
	def CompanyInfo_task(self, agent, company):
		return Task(
			description=dedent(f"""\
				Conduct comprehensive research on the company named {company}. Gather information about its website, founded date of company, its founders, headquarter location of the company, industry of the company and its subsidiaries.
				Company Name : {company} """),
			expected_output=dedent("""\
				A detailed report summarizing key findings about the company,its website, founded date, founders, headquarter,industry and its subsidiaries."""),
			async_execution=True,
			agent=agent
		)

	def WebsiteAnalysis_task(self, agent, company):
		return Task(
			description=dedent(f"""\
				Uncover information about the website of company named {company}. find its official website and its structure, review the content of the website, make analysis on its metadata, find the technology stack used in the website take the help fo tools like buitWith to find technology stack, also find the SSL/TLS configuration.

                Company Name: {company}"""),
			expected_output=dedent("""\
				A comprehensive report on the company's Website Structure,content review, metadata analysis, technology stack and SSL/TLS configuration."""),
			async_execution=True,
			agent=agent
		)


	def NetworkAnalysis_task(self, agent, company):
		return Task(
			description=dedent(f"""\
				Uncover information about the Domain name and Network of company named {company}.
        find its domain registration details using tool like 'whois',
        find its dns records using tools like 'DNSdumpster',
        find tis subdomains using dns tools and google dorking techniques,
        find its ip addrress associated with domain and
        find different network services using shodan.

        Company Name: {company}"""),
			expected_output=dedent("""\
				A comprehensive report on the company's domain registration details, dns records, subdomains, IP addresses and network services."""),
			async_execution=True,
			agent=agent
		)

	def SocialMediaAndContact_task(self, agent, company):
		return Task(
			description=dedent(f"""\
				Gather detailed information about the social media presence and contact details of the company named {company}.
                Find links to its profiles on platforms like LinkedIn, Facebook, Twitter, GitHub, Instagram, and others.
                Additionally, gather comprehensive contact information including phone numbers, email addresses, and contact details of key personnel if available.
                Company Name: {company}"""),
			expected_output=dedent("""\
				A comprehensive report on the company's social media profiles and contact details, including phone numbers, email addresses, and contact information of key personnel."""),
			async_execution=True,
			agent=agent
		)

	def SearchEngineIntelligence_task(self, agent, company):
		return Task(
			description=dedent(f"""\
            Use Google Dorking techniques to uncover hidden information like PDFs and confidential files about the company named {company}.
            Also, find recent news articles about the company, analyze them, and derive conclusions.
            Company Name: {company}"""),
			expected_output=dedent("""\
            A report on hidden information uncovered using Google Dorking and a summary of recent news articles with analysis and conclusions."""),
			async_execution=True,
			agent=agent
		)

	def BusinessInformation_task(self, agent, company):
		return Task(
			description=dedent(f"""\
            Gather comprehensive business information about the company named {company}.
            This includes an overview from business directories like Bloomberg, Crunchbase, LinkedIn, financial information, key personnel details, and information about company partnerships.
            Company Name: {company}"""),
			expected_output=dedent("""\
            A detailed report summarizing the company's business information, including company overview, financial data, key personnel, and partnerships"""),
			async_execution=True,
			agent=agent
		)

	def RegulatoryLegalTechnicalFootprint_task(self, agent, company):
		return Task(
			description=dedent(f"""\
            Gather information about the regulatory filings, legal issues, security posture, and email patterns of the company named {company}.
            This includes checking filings with bodies like the SEC, identifying ongoing or past legal issues, assessing vulnerabilities using tools like Shodan, and finding company email patterns using tools like Hunter.io.
            Company Name: {company}"""),
			expected_output=dedent("""\
				A comprehensive report on the company's regulatory filings, legal issues, security posture, and email patterns."""),
			async_execution=True,
			agent=agent
		)

	def IntellectualProperty_task(self, agent, company):
		return Task(
			description=dedent(f"""\
            Gather information about the intellectual property of the company named {company}.
            This includes any patents registered by the company, trademarks, and significant copyrights.
            Company Name: {company}"""),
			expected_output=dedent("""\
				A detailed report on the company's intellectual property, including patents, trademarks, and copyrights."""),
			async_execution=True,
			agent=agent
		)

	def EmployeeHiringInformation_task(self, agent, company):
		return Task(
			description=dedent(f"""\
            Gather information about current job listings and employee reviews for the company named {company}.
            This includes job openings from the company’s career page and other job boards, as well as employee reviews from sites like Glassdoor.
            Company Name: {company}"""),
			expected_output=dedent("""\
				A comprehensive report on the company's current job listings and employee reviews"""),
			async_execution=True,
			agent=agent
		)

	def CommunityPublicPerception_task(self, agent, company):
		return Task(
			description=dedent(f"""\
            Gather customer reviews and forum discussions related to the company named {company}.
            This includes reviews from websites like Trustpilot and Google Reviews, and mentions on forums like Reddit and industry-specific forums.
            Company Name: {company}"""),
			expected_output=dedent("""\
				A report on the company's community and public perception, including customer reviews and forum discussions."""),
			async_execution=True,
			agent=agent
		)

	def OSINTReportGenerator_task(self, agent, company):
		return Task(
      description=dedent(f"""\
            Compile all the gathered information, including Company Information, Website Analysis, Domain and Network Analysis, Social Media and Contact Information, Search Engine Intelligence, Business Information, Regulatory and Legal Information, Technical Footprint, Intellectual Property, Employee and Hiring Information, Community and Public Perception, and Dark Web Mentions, into a concise and comprehensive OSINT report for the company.
            Ensure the report is detailed, well-structured, and provides valuable insights.
            Company Name: {company}"""),
      expected_output=dedent("""\
                A detailed and well-structured OSINT report for the company, including sections on Company Information, Website Analysis, Domain and Network Analysis, Social Media and Contact Information, Search Engine Intelligence, Business Information, Regulatory and Legal Information, Technical Footprint, Intellectual Property, Employee and Hiring Information, Community and Public Perception, and Dark Web Mentions."""),
      agent=agent
    )
     

from crewai import Crew

tasks = OsintAnalysisTask()
agents = OsintAgents()

print("## Welcome to Osint Anaysis of Company")
print('-------------------------------')
company = input("Enter the name of company\n")

# Create Agents
Information_agent = agents.CompanyInfo_agent()
WebsiteAnalyst = agents.WebsiteAnalysis_agent()
NetworkAnalyst = agents.NetworkAnalysis_agent()
SocialMediaAndContact = agents.SocialMediaAndContact_agent()
SearchEngineIntelligence = agents.SearchEngineIntelligence_agent()
BusinessInformation = agents.BusinessInformation_agent()
RegulatoryLegalTechnicalFootprint = agents.RegulatoryLegalTechnicalFootprint_agent()
IntellectualProperty = agents.IntellectualProperty_agent()
EmployeeHiringInformation = agents.EmployeeHiringInformation_agent()
CommunityPublicPerception = agents.CommunityPublicPerception_agent()
OsintReporter = agents.OSINTReportGenerator_agent()


# Create Tasks
information = tasks.CompanyInfo_task(Information_agent , company)
website_analysis = tasks.WebsiteAnalysis_task(WebsiteAnalyst, company)
network_analysis = tasks.NetworkAnalysis_task(NetworkAnalyst, company)
social_media_and_contact = tasks.SocialMediaAndContact_task(SocialMediaAndContact, company)
search_engine_intelligence = tasks.SearchEngineIntelligence_task(SearchEngineIntelligence, company)
business_information = tasks.BusinessInformation_task(BusinessInformation, company)
regulatory_legal_technical_footprint = tasks.RegulatoryLegalTechnicalFootprint_task(RegulatoryLegalTechnicalFootprint, company)
intellectual_property = tasks.IntellectualProperty_task(IntellectualProperty, company)
employee_hiring_information = tasks.EmployeeHiringInformation_task(EmployeeHiringInformation, company)
community_public_perception = tasks.CommunityPublicPerception_task(CommunityPublicPerception, company)
summary_and_briefing = tasks.OSINTReportGenerator_task(OsintReporter, company)

# create context
summary_and_briefing.context = [
    information,
    website_analysis,
    network_analysis,
    social_media_and_contact,
    search_engine_intelligence,
    business_information,
    regulatory_legal_technical_footprint,
    intellectual_property,
    employee_hiring_information,
    community_public_perception
    ]

# Create Crew responsible for Copy
crew = Crew(
	agents=[
		Information_agent,
		WebsiteAnalyst,
    NetworkAnalyst,
    SocialMediaAndContact,
    SearchEngineIntelligence,
    BusinessInformation,
    RegulatoryLegalTechnicalFootprint,
    IntellectualProperty,
    EmployeeHiringInformation,
    CommunityPublicPerception,
		OsintReporter
	],
	tasks=[
		information,
		website_analysis,
    network_analysis,
    social_media_and_contact,
    search_engine_intelligence,
    business_information,
    regulatory_legal_technical_footprint,
    intellectual_property,
    employee_hiring_information,
    community_public_perception,
    summary_and_briefing
	]
)

result = await crew.kickoff_async()

final_output = result.raw if hasattr(result, "raw") else str(result)

# Print results
print("\n\n################################################")
print("## Here is the result")
print("################################################\n")
print(final_output)

from IPython.display import display, Markdown
display(Markdown(final_output))

Ollama server erreichbar: http://localhost:11434/v1/models
Verfügbare Modelle: []
## Welcome to Osint Anaysis of Company
-------------------------------


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Company Information Specialist                                                                          │
│                                                                                                                 │
│  Task: Conduct comprehensive research on the company named "Wechsler Information Solution". Gather information  │
│  about its website, founded date of company, its founders, headquarter location of the company, industry of     │
│  the company and its subsidiaries.                                                                              │
│  Company Name : "Wechsler Information Solution"                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Domain and Network Analyst                                                                              │
│                                                                                                                 │
│  Task:                           Uncover information about the Domain name and Network of company named         │
│  "Wechsler Information Solution".                                                                               │
│          find its domain registration details using tool like 'whois',                                          │
│          find its dns records using tools like 'DNSdumpster',                                                   │
│          find tis subdomains using dns tools and google dorking techniques,                                     │
│          find its ip addrress associated with domain and                                                        │
│          find different network services using shodan.                                                          │
│                                                                                                                 │
│          Company Name: "Wechsler Information Solution"                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Website Analyst                                                                                         │
│                                                                                                                 │
│  Task:                           Uncover information about the website of company named "Wechsler Information   │
│  Solution". find its official website and its structure, review the content of the website, make analysis on    │
│  its metadata, find the technology stack used in the website take the help fo tools like buitWith to find       │
│  technology stack, also find the SSL/TLS configuration.                                                         │
│                                                                                                                 │
│                  Company Name: "Wechsler Information Solution"                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Social Media and Contact Information Specialist                                                         │
│                                                                                                                 │
│  Task:                           Gather detailed information about the social media presence and contact        │
│  details of the company named "Wechsler Information Solution".                                                  │
│                  Find links to its profiles on platforms like LinkedIn, Facebook, Twitter, GitHub, Instagram,   │
│  and others.                                                                                                    │
│                  Additionally, gather comprehensive contact information including phone numbers, email          │
│  addresses, and contact details of key personnel if available.                                                  │
│                  Company Name: "Wechsler Information Solution"                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: OSINT Report Generator                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│   Here's an overview of the information provided on Wechsler Information Solution:                              │
│                                                                                                                 │
│  1. Company Overview:                                                                                           │
│     - Wechsler Information Solution is a San Francisco-based company founded in 2014 by Dan Wechsler,           │
│  specializing in data analytics for law firms through eDiscovery and managed document review services.          │
│     - The company has reported partnerships with industry leaders such as Relativity and The eDISCOVERY         │
│  Institute to expand its reach and improve its service offerings.                                               │
│     - Regulatory filings for the company are not publicly available, possibly indicating it is private or a     │
│  subsidiary of another larger organization. No ongoing legal cases were found during the search.                │
│                                                                                                                 │
│  2. Contact Information:                                                                                        │
│     - Email patterns associated with Wechsler Information Solution: [firstname]@wechslerinfo.com,               │
│  support@wechslerinfo.com, sales@wechslerinfo.com, info@wechslerinfo.com                                        │
│       (Please note that the actual email addresses may not be accurate due to privacy concerns and are          │
│  available only in a limited form.)                                                                             │
│     - The company's career page: https://www.wechsleris.com/careers                                             │
│                                                                                                                 │
│  3. Intellectual Property:                                                                                      │
│     - A comprehensive search for patents, trademarks, and copyrights held by Wechsler Information Solution      │
│  will be conducted following these steps:                                                                       │
│       1. Search for patents registered by Wechsler Information Solution                                         │
│       2. Find similar patents to those identified in step 1 that might extend beyond the initial search result  │
│  boundaries                                                                                                     │
│       3. Search for registered trademarks owned by Wechsler Information Solution                                │
│       4. Find similar trademarks to those identified in step 3 that might extend beyond the initial search      │
│  result boundaries                                                                                              │
│       5. Search for copies of any significant copyrights owned by Wechsler Information Solution specifically    │
│  mentioning "Wechsler Information Solution" as the copyright holder or publisher                                │
│       6. Investigate the obtained copyrights further to find out if they have been officially registered, thus  │
│  ensuring their legal protection and recognition.      



################################################
## Here is the result
################################################

 Here's an overview of the information provided on Wechsler Information Solution:

1. Company Overview:
   - Wechsler Information Solution is a San Francisco-based company founded in 2014 by Dan Wechsler, specializing in data analytics for law firms through eDiscovery and managed document review services.
   - The company has reported partnerships with industry leaders such as Relativity and The eDISCOVERY Institute to expand its reach and improve its service offerings.
   - Regulatory filings for the company are not publicly available, possibly indicating it is private or a subsidiary of another larger organization. No ongoing legal cases were found during the search.

2. Contact Information:
   - Email patterns associated with Wechsler Information Solution: [firstname]@wechslerinfo.com, support@wechslerinfo.com, sales@wechslerinfo.com, info@wechslerinfo.com
     

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Here's an overview of the information provided on Wechsler Information Solution:

1. Company Overview:
   - Wechsler Information Solution is a San Francisco-based company founded in 2014 by Dan Wechsler, specializing in data analytics for law firms through eDiscovery and managed document review services.
   - The company has reported partnerships with industry leaders such as Relativity and The eDISCOVERY Institute to expand its reach and improve its service offerings.
   - Regulatory filings for the company are not publicly available, possibly indicating it is private or a subsidiary of another larger organization. No ongoing legal cases were found during the search.

2. Contact Information:
   - Email patterns associated with Wechsler Information Solution: [firstname]@wechslerinfo.com, support@wechslerinfo.com, sales@wechslerinfo.com, info@wechslerinfo.com
     (Please note that the actual email addresses may not be accurate due to privacy concerns and are available only in a limited form.)
   - The company's career page: https://www.wechsleris.com/careers

3. Intellectual Property:
   - A comprehensive search for patents, trademarks, and copyrights held by Wechsler Information Solution will be conducted following these steps:
     1. Search for patents registered by Wechsler Information Solution
     2. Find similar patents to those identified in step 1 that might extend beyond the initial search result boundaries
     3. Search for registered trademarks owned by Wechsler Information Solution
     4. Find similar trademarks to those identified in step 3 that might extend beyond the initial search result boundaries
     5. Search for copies of any significant copyrights owned by Wechsler Information Solution specifically mentioning "Wechsler Information Solution" as the copyright holder or publisher
     6. Investigate the obtained copyrights further to find out if they have been officially registered, thus ensuring their legal protection and recognition.

4. Job Listings:
   - Job openings at Wechsler Information Solution can be found by searching its career page and various job boards (LinkedIn, Indeed, Glassdoor):

```
search("https://www.wechsleris.com/careers")
find_similar("https://www.linkedin.com/company/wechsler-information-solution")
find_similar("https://www.indeed.com/cmp/Wechsler-Information-Solution")
find_similar("https://www.glassdoor.com/Job/Wechsler-Information-Solution-Jobs-E3208519.htm")
```
   - Once the relevant job listings have been found, their contents can be extracted using `get_contents`:

```
get_contents(ids=[urls returned from search])
```

5. Customer Reviews:
   - Trustpilot reviews for Wechsler Information Solution:

```
search(query="Wechsler Information Solution site:trustpilot.com")
```
   - Google Reviews:

```
search(query="Wechsler Information Solution site:google.com/maps")
```
   - Reddit mentions:

```
search(query="Wechsler Information Solution site:reddit.com")
```
   - Searches for industry-specific forum discussions related to "Wechsler Information Solution":

```
search(query="Wechsler Information Solution site:[insert industry forum URL here]")
replace [insert industry forum URL here] with the URL of an appropriate industry forum that relates to IT or data services.
```

In [ ]:

import re
from datetime import datetime

safe_company = re.sub(r'[^\w\-]', '_', company)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"osint_report_{safe_company}_{timestamp}.md"

with open(filename, "w", encoding="utf-8") as f:
    f.write(f"# OSINT Report: {company}\n\n")
    f.write(f"*Erstellt am: {datetime.now().strftime('%d.%m.%Y %H:%M:%S')}*\n\n")
    f.write("---\n\n")
    f.write(final_output)

print(f"Markdown-Bericht gespeichert: {filename}")
